# loom multi-agent build on Colab

Multi-agent is only justified where one agent cannot do the job in one shot, so this notebook compares both honestly. An orchestrator splits a spec of four tricky python modules (tokenizer, intervals, TTL/LRU cache, money rounding) into worker subtasks. Workers write files into a persistent sandbox with loom's `write_file` tool. The harness runs **hidden** acceptance tests that live outside the sandbox and never appear in a prompt; workers only see failing test names and failure kinds (up to 3 review rounds, so at most 2 rework rounds). A single agent gets the same spec and the same hidden tests in one shot; its pass count is reported next to the multi-agent result.

A failed orchestrator, worker or reviewer fails the run (never PASS). Each agent retries a bounded number of times, then falls back to the next provider; provider errors are summarised per agent with keys redacted.

Add `ANTHROPIC_API_KEY` and/or `GEMINI_API_KEY` in the Colab Secrets panel (key icon, enable notebook access) for real models. Keys are read only via `google.colab.userdata`, passed to the loom child process, and never printed. With no key the offline mock path runs: loom's mock provider cannot call tools, so the harness writes scripted file payloads (two flawed modules fixed in round 2). Mock numbers are scripted, not evidence about any model.

## 1. Install nexus-lite

In [ ]:
!pip install -q git+https://github.com/choilab-jefferson/nexus.lite.git pytest

## 2. Run (auto: real providers if secrets exist, else mock)
The sandbox directory is kept and its path printed. The first real run downloads loom (checksum-verified release, falls back to a source build). Add `--json-out run.json` to keep rounds, per-agent status/time/error, hidden pass counts and baseline numbers.

In [ ]:
from nexus_lite.loom_multi_agent import main
main([])

Force the offline path: `main(['--providers', 'mock'])`. Failure injection, e.g. a wrong key for one worker so retry/fallback run: `main(['--providers', 'anthropic,gemini', '--inject-bad-key', 'worker-2'])`; a short timeout: `'--inject-timeout', 'reviewer=1'`. Repeat runs: `'--runs', '3'`. Baseline self-repair, reported separately: `'--baseline-repair', '3'`.